# 📜 El Manuscrito del Sabio: Sistema RAG Completo
> *El Sabio despliega el pergamino sagrado sobre la mesa de neón: "Observa bien, joven Ninja. Aquí yace la restauración total del Gran Archivo. Cada línea de código es un sello que sostiene la Tríada."*

**Objetivo:** Cargar e inicializar las librerías necesarias de LangChain, OpenAI y ChromaDB para construir la canalización RAG completa.

In [8]:
# 📦 Instalación de las dependencias del proyecto
!pip install -q \
    "chromadb==0.5.23" \
    "opentelemetry-api==1.29.0" \
    "opentelemetry-sdk==1.29.0" \
    "opentelemetry-proto==1.29.0" \
    "langchain==0.3.27" \
    "langchain-community==0.3.27" \
    "langchain-core==0.3.72" \
    "langchain-text-splitters==0.3.9" \
    "langchain-huggingface" \
    "sentence-transformers" \
    "pypdf"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB

In [23]:
# 🥷 Carga de librerías esenciales

# Herramientas generales de Python
import os
from typing import Optional, List

# PyTorch y Transformers para cargar y ejecutar el modelo Qwen
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 📄 Herramientas para leer el PDF y dividirlo en fragmentos
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 🔮 Herramientas para generar embeddings y almacenar los vectores
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 🛡️ Herramientas necesarias para construir la cadena RAG
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# 🧙 Herramientas para crear el prompt y conectar Qwen con LangChain
from langchain_core.prompts import PromptTemplate
from langchain_core.language_models.llms import LLM

# Comprobación de que las librerías se hayan cargado correctamente
print("✅ Pergaminos de librerías listos y funcionales.")

# Se comprueba si el entorno dispone de una GPU
# Si devuelve True, Qwen podrá ejecutarse utilizando CUDA.
print(f"🔥 GPU disponible: {torch.cuda.is_available()}")

✅ Pergaminos de librerías listos y funcionales.
🔥 GPU disponible: True


### 📜 Paso 1: Lectura del Archivo Sagrado
> *El Sabio señala el documento `teoria_rag.pdf`: "Extraemos el conocimiento puro del pergamino principal para procesarlo en nuestra memoria."*

In [3]:
# 🥷 Carga y extracción del PDF

# Se indica la ubicación del documento que se quiere utilizar
# como fuente de conocimiento para nuestro sistema RAG.
path_pdf = "docs/teoria_rag.pdf"

# Se crea el cargador de PDF de LangChain.
# PyPDFLoader se encarga de leer el contenido del archivo PDF.
loader = PyPDFLoader(path_pdf)

# Se extrae el texto del PDF.
# Cada página se almacena como un documento independiente,
# conservando también información como el número de página.
docs = loader.load()

# Se muestran cuántas páginas fueron cargadas correctamente.
print(f"📄 Páginas cargadas exitosamente: {len(docs)}")

📄 Páginas cargadas exitosamente: 9


### 🧩 Paso 2: La Fragmentación (Chunking)
> *El Sabio advierte: "Fragmentamos el texto en bloques de 1000 caracteres con un solapamiento de 100 para preservar el contexto entre límites."*

In [18]:
# 🥷 Segmentación del texto en chunks

# Se crea el separador de texto.
# Su función es dividir el contenido del PDF en fragmentos más pequeños
# para que posteriormente puedan convertirse en embeddings y ser recuperados.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,                   # Tamaño máximo aproximado de cada fragmento
    chunk_overlap=200                  # Caracteres compartidos entre fragmentos consecutivos
)

# Se dividen las páginas del PDF en fragmentos.
# Cada fragmento conserva también los metadatos del documento original.
splits = text_splitter.split_documents(docs)

# Se muestra cuántos fragmentos se generaron.
print(f"🧩 Fragmentos generados: {len(splits)}")

🧩 Fragmentos generados: 10


### 🔮 Paso 3: Encantamiento de Vectorización y ChromaDB
> *"Transformamos las palabras en vectores densos mediante el modelo de embeddings y los almacenamos en la base de datos vectorial ChromaDB."*

In [5]:
# 🥷 Generación de embeddings y almacenamiento vectorial

# Se carga un modelo de embeddings.
# Este modelo transforma cada fragmento de texto en un vector numérico
# que representa su significado semántico.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Se crea la base de datos vectorial ChromaDB.
# Los fragmentos generados en el Paso 2 se convierten en vectores
# mediante el modelo de embeddings y se almacenan en Chroma.
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="santuario_rag"
)

# Se crea el retriever, encargado de buscar los fragmentos
# más relevantes cuando el usuario realiza una pregunta.
# k=3 significa que recuperaremos los 3 fragmentos más relevantes.
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("🔮 Santuario vectorial inicializado y listo para búsquedas.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🔮 Santuario vectorial inicializado y listo para búsquedas.


### 🧙 Paso 4: Invocación del Sabio Local (LLM)
> "Cargamos un modelo de lenguaje Qwen de código abierto directamente en el entorno de ejecución. El modelo se ejecuta localmente, utilizando la GPU cuando está disponible, y se configura para generar respuestas de forma determinista sin depender de la API de OpenAI."

In [19]:
# Se indica el modelo de lenguaje que utilizaremos como generador
# de respuestas dentro de nuestra arquitectura RAG.
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Se carga el tokenizer.
# El cual se encarga de convertir el texto en tokens que el modelo puede procesar.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Se carga el modelo Qwen para generación de texto.
model = AutoModelForCausalLM.from_pretrained(
    model_name
)

# Se comprueba si se tiene una GPU disponible.
# Si existe, se usa CUDA para acelerar la generación.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Se mueve el modelo al dispositivo seleccionado.
model = model.to(device)

# Se configura la generación de forma determinista.
# Al desactivar el muestreo, el modelo produce respuestas más
# consistentes ante la misma entrada.
model.generation_config.do_sample = False

# Se eliminan los parámetros relacionados con el muestreo,
# ya que no se utilizan cuando do_sample=False.
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

# Eso evita que exista un límite de longitud heredado
# que pueda entrar en conflicto con max_new_tokens.
model.generation_config.max_length = None

print(f"🤖 Sabio Qwen cargado en: {device}")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🤖 Sabio Qwen cargado en: cuda


### 🧙 Paso 5: El Puente del Sabio con LangChain
> "Construimos un adaptador que conecta el modelo Qwen ejecutado localmente con LangChain. Este puente recibe el prompt generado por la cadena RAG, lo transforma al formato de conversación de Qwen, ejecuta la generación en la GPU disponible y devuelve únicamente la respuesta generada por el Sabio."

In [20]:
class QwenLocalLLM(LLM):

    # Identificamos el tipo de modelo que estamos utilizando
    @property
    def _llm_type(self) -> str:
        return "qwen_local"

    # Este método se ejecuta cada vez que LangChain necesita obtener una respuesta del modelo Qwen.
    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs
    ) -> str:

        # Construimos los mensajes que recibirá Qwen.
        # El mensaje "system" define el comportamiento del modelo.
        # El mensaje "user" contiene el prompt con el contexto y la pregunta.
        messages = [
            {
                "role": "system",
                "content": (
                    "Eres un asistente de preguntas y respuestas. "
                    "Responde únicamente utilizando el contexto proporcionado."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        # Se convierten los mensajes al formato de conversación que espera específicamente el modelo Qwen
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Se convierte el texto en tokens y tensores de PyTorch.
        # .to(device) coloca los datos en la GPU o CPU correspondiente.
        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(device)

        # Se desactiva el cálculo de gradientes porque solo se utiliza el modelo para
        # generar una respuesta, no para entrenarlo.
        with torch.no_grad():

            # Se ejecuta Qwen para generar la respuesta.
            outputs = model.generate(
                **inputs,

                # Se limita la respuesta a un máximo de 120 tokens.
                max_new_tokens=120,

                # Se desactiva el muestreo para obtener respuestas deterministas y reproducibles.
                do_sample=False,

                # Se indica qué token representa el final de la generación.
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Se elimina de la salida los tokens correspondientes
        # al prompt original y conservamos solamente la respuesta generada.
        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

        # Se convierten los tokens generados nuevamente a texto.
        answer = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

        # Se eliminan espacios innecesarios y se devuelve la respuesta.
        return answer.strip()

# Se crea una instancia del adaptador para que LangChain
# pueda utilizar nuestro modelo Qwen.
llm = QwenLocalLLM()

print("🧙 Sabio conectado correctamente con LangChain.")

🧙 Sabio conectado correctamente con LangChain.


### 🛡️ Paso 6: La Cadena del Oráculo (RAG)
> "Unimos las tres piezas del sistema: el recuperador de ChromaDB, el contexto obtenido de los documentos y el Sabio Qwen. El prompt establece las reglas para que Qwen responda únicamente con la información recuperada, evitando conocimientos externos e invenciones. Finalmente, create_retrieval_chain construye la cadena RAG completa que recupera el contexto y genera la respuesta."

In [21]:
# 🛡️ Paso 6: La Cadena del Oráculo (RAG)

# Se definen las instrucciones que recibirá el modelo.
template = """Responde la pregunta utilizando únicamente el contexto.

CONTEXTO:
{context}

PREGUNTA:
{input}

REGLAS:
- No inventes información.
- No utilices conocimientos externos.
- Conserva los nombres técnicos exactamente como aparecen en el contexto.
- Si la pregunta pide tres elementos, proporciona exactamente tres.
- Si la respuesta no aparece en el contexto, responde:
Información no encontrada en el Gran Archivo.

RESPUESTA:
"""

# Se el prompt indicando las variables que LangChain
# sustituirá automáticamente:
# {context} → documentos recuperados por ChromaDB
# {input}   → pregunta realizada por el usuario
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "input"]
)

# Se crea la cadena encargada de combinar los documentos recuperados
# y enviarlos junto con el prompt al modelo Qwen.
question_answer_chain = create_stuff_documents_chain(
    llm,
    prompt
)

# Se crea la cadena RAG completa.
# Primero recupera los documentos relevantes mediante el retriever
# y después utiliza Qwen para generar la respuesta.
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

print("🛡️ Cadena RAG ensamblada con prompt estricto.")

🛡️ Cadena RAG ensamblada con prompt estricto.


### ⚡ Paso 7: La Prueba de la Tríada
> "Ponemos a prueba el sistema RAG realizando una consulta sobre la Tríada RAG. El retriever busca los fragmentos relevantes del documento y el Sabio Qwen genera la respuesta utilizando únicamente el contexto recuperado."

In [22]:
# ⚡ Paso 7: La Prueba de la Tríada

# Pregunta que se quiere realizar al sistema RAG
pregunta = "¿Cuáles son las tres métricas de evaluación de la Tríada RAG?"


# Se ejecuta la cadena RAG:
# 1. El retriever busca los fragmentos más relevantes.
# 2. El prompt incorpora esos fragmentos como contexto.
# 3. Qwen genera la respuesta utilizando ese contexto.
response = rag_chain.invoke({
    "input": pregunta
})

# Se muestra únicamente la respuesta generada por el modelo
print("🤖 Respuesta del Sabio:\n")
print(response["answer"])

🤖 Respuesta del Sabio:

Las tres métricas de evaluación de la Tríada RAG son:

1. Context Relevance (Relevancia del Contexto)
2. Groundedness / Faithfulness (Fidelidad)
3. Answer Relevance (Relevancia de la Respuesta)


### 📝 Conclusiones y Demostración Docente
> *El Sabio cierra el libro con satisfacción: "Has presenciado la restauración total. La Tríada RAG garantiza respuestas precisas y libres de alucinaciones."*

### 🎯 Resumen Técnico de la Solución

1. **Garantía de Fidelidad (Groundedness):** El prompt del sistema prohíbe explícitamente el uso de conocimiento externo al contexto recuperado.
2. **Optimizaciones Aplicadas:** Se usó un `chunk_size` de 1000 caracteres con un solapamiento de 100 para evitar pérdidas semánticas en las fronteras del texto.
3. **Flujo de Ejecución:** Lectura de PDF ➔ Splitter ➔ Embeddings ➔ ChromaDB ➔ Chain RAG.